# AethyxLM v3 — production resume on Kaggle TPU

This notebook resumes the **48K-vocabulary, ~138M-parameter** v3 run on every
core of a Kaggle TPU VM. It keeps the original global batch of 32, restores the
CUDA-produced checkpoint on CPU before moving optimizer state to XLA, saves
portable CPU checkpoints every 1,000 steps, and backs them up persistently.


In [ ]:
from pathlib import Path
import base64, gc, json, os, re, signal, subprocess, sys, time

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this notebook on Kaggle.')
os.environ.update({
    'PJRT_DEVICE': 'TPU', 'TOKENIZERS_PARALLELISM': 'false',
    'PYTHONUNBUFFERED': '1', 'OMP_NUM_THREADS': '1',
})
REPO_URL = 'https://github.com/aethyx-ai/AethyxLM.git'
REPO_ROOT = Path('/kaggle/working/aethyxlm-v3-repo')
PROJECT_ROOT = REPO_ROOT / 'AethyxLM'
OUTPUT_ROOT = Path('/kaggle/working/aethyxlm-v3-output')
CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints'
LOG_ROOT = OUTPUT_ROOT / 'logs'
CONFIG_ROOT = OUTPUT_ROOT / 'configs'
for path in (CHECKPOINT_ROOT, LOG_ROOT, CONFIG_ROOT, CHECKPOINT_ROOT / 'milestones'):
    path.mkdir(parents=True, exist_ok=True)

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
elif REPO_ROOT.exists():
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git checkout.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

# Install the tested TPU implementation into this private runtime without
# requiring an intermediate public repository commit.
embedded = json.loads('{"train.py":"...","train_xla.py":"IiIiQWxsLWNvcmUgUHlUb3JjaC9YTEEgbGF1bmNoZXIgZm9yIEFldGh5eExNIHRyYWluaW5nLiIiIgoKaW1wb3J0IG9zCmltcG9ydCBzeXMKCgpkZWYgX3dvcmtlcihfaW5kZXg6IGludCwgZm9yd2FyZGVkOiBsaXN0W3N0cl0pOgogICAgIyBYTEEgZGV2aWNlcyBtdXN0IG9ubHkgYmUgYWNxdWlyZWQgYmVsb3cgdG9yY2hfeGxhLmxhdW5jaCgpLgogICAgc3lzLmFyZ3YgPSBbInRyYWluLnB5IiwgKmZvcndhcmRlZCwgIi0tZGV2aWNlIiwgInhsYSJdCiAgICBmcm9tIHRyYWluIGltcG9ydCBtYWluCgogICAgbWFpbigpCgoKZGVmIG1haW4oKToKICAgIGZvcndhcmRlZCA9IHN5cy5hcmd2WzE6XQogICAgIyBLYWdnbGUgc2V0cyB0aGVzZSB0byBzaW5nbGUtaG9zdCBwbGFjZWhvbGRlcnMgKGZvciBleGFtcGxlLCAibG9jYWwiKS4KICAgICMgUEpSVCBtdWx0aXByb2Nlc3NpbmcgbWlzdGFrZXMgdGhlbSBmb3IgYW4gaW5jb21wbGV0ZSBtdWx0aS1ob3N0IHRvcG9sb2d5LgogICAgb3MuZW52aXJvbi5wb3AoIlRQVV9QUk9DRVNTX0FERFJFU1NFUyIsIE5vbmUpCiAgICBvcy5lbnZpcm9uLnBvcCgiQ0xPVURfVFBVX1RBU0tfSUQiLCBOb25lKQogICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJQSlJUX0RFVklDRSIsICJUUFUiKQoKICAgIGltcG9ydCB0b3JjaF94bGEKCiAgICAjIFRoZSBzdXBwb3J0ZWQgbGF1bmNoZXIgZGlzY292ZXJzIGFuZCBzdGFydHMgZXZlcnkgYWRkcmVzc2FibGUgVFBVIHdvcmtlci4KICAgICMgTm90aGluZyBtYXkgYWNxdWlyZSBhbiBYTEEgZGV2aWNlIGJlZm9yZSB0aGlzIGNhbGwuCiAgICB0b3JjaF94bGEubGF1bmNoKF93b3JrZXIsIGFyZ3M9KGZvcndhcmRlZCwpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK","training/trainer.py":"...","model/attention.py":"IiIiRWZmaWNpZW50IGNhdXNhbCBzZWxmLWF0dGVudGlvbiBmb3IgQWV0aHl4TE0uCgpTdXBwb3J0cyBtdWx0aS1oZWFkIGFuZCBncm91cGVkLXF1ZXJ5IGF0dGVudGlvbiwgUHlUb3JjaCBTRFBBL0ZsYXNoIGtlcm5lbHMsCm9wdGlvbmFsIGZ1c2VkIFFLViBwcm9qZWN0aW9ucywgUm9QRSwgUUsgbm9ybWFsaXphdGlvbiwgYW5kIGluZmVyZW5jZSBLViBjYWNoZS4KIiIiCgpmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwsIFR1cGxlCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCmZyb20gbW9kZWwuY29uZmlnIGltcG9ydCAoCiAgICBDT05URVhUX0xFTkdUSCwKICAgIERST1BPVVQsCiAgICBFTUJFRF9ESU0sCiAgICBOVU1fSEVBRFMsCiAgICBQT1NJVElPTl9FTkNPRElORywKICAgIFJPUEVfQkFTRSwKICAgIFJPUEVfTUFYX1NFUV9MRU4sCiAgICBVU0VfQklBUywKKQpmcm9tIG1vZGVsLmxheWVycyBpbXBvcnQgTGluZWFyCmZyb20gbW9kZWwubW9kdWxlcy5yb3BlIGltcG9ydCBSb3RhcnlFbWJlZGRpbmcKCgpLVkNhY2hlID0gVHVwbGVbdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3JdCgoKY2xhc3MgTXVsdGlIZWFkU2VsZkF0dGVudGlvbihubi5Nb2R1bGUpOgogICAgIiIiQ2F1c2FsIHNlbGYtYXR0ZW50aW9uIHdpdGggb3B0aW9uYWwgZ3JvdXBlZCBrZXkvdmFsdWUgaGVhZHMuIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgZW1iZWRfZGltOiBpbnQgPSBOb25lLAogICAgICAgIG51bV9oZWFkczogaW50ID0gTm9uZSwKICAgICAgICBudW1fa3ZfaGVhZHM6IGludCA9IE5vbmUsCiAgICAgICAgZHJvcG91dDogZmxvYXQgPSBOb25lLAogICAgICAgIGNvbnRleHRfbGVuZ3RoOiBpbnQgPSBOb25lLAogICAgICAgIHVzZV9iaWFzOiBib29sID0gTm9uZSwKICAgICAgICBwb3NpdGlvbl9lbmNvZGluZzogc3RyID0gTm9uZSwKICAgICAgICByb3BlX2Jhc2U6IGZsb2F0ID0gTm9uZSwKICAgICAgICByb3BlX21heF9zZXFfbGVuOiBpbnQgPSBOb25lLAogICAgICAgIHJvcGVfc2NhbGluZ19mYWN0b3I6IGZsb2F0ID0gMS4wLAogICAgICAgIGZ1c2VkX3FrdjogYm9vbCA9IEZhbHNlLAogICAgICAgIHVzZV9zZHBhOiBib29sID0gVHJ1ZSwKICAgICAgICBxa19ub3JtOiBib29sID0gRmFsc2UsCiAgICAgICAgc2xpZGluZ193aW5kb3c6IE9wdGlvbmFsW2ludF0gPSBOb25lLAogICAgKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBlbWJlZF9kaW0gPSBFTUJFRF9ESU0gaWYgZW1iZWRfZGltIGlzIE5vbmUgZWxzZSBlbWJlZF9kaW0KICAgICAgICBudW1faGVhZHMgPSBOVU1fSEVBRFMgaWYgbnVtX2hlYWRzIGlzIE5vbmUgZWxzZSBudW1faGVhZHMKICAgICAgICBudW1fa3ZfaGVhZHMgPSBudW1faGVhZHMgaWYgbnVtX2t2X2hlYWRzIGlzIE5vbmUgZWxzZSBudW1fa3ZfaGVhZHMKICAgICAgICBkcm9wb3V0ID0gRFJPUE9VVCBpZiBkcm9wb3V0IGlzIE5vbmUgZWxzZSBkcm9wb3V0CiAgICAgICAgY29udGV4dF9sZW5ndGggPSBDT05URVhUX0xFTkdUSCBpZiBjb250ZXh0X2xlbmd0aCBpcyBOb25lIGVsc2UgY29udGV4dF9sZW5ndGgKICAgICAgICB1c2VfYmlhcyA9IFVTRV9CSUFTIGlmIHVzZV9iaWFzIGlzIE5vbmUgZWxzZSB1c2VfYmlhcwogICAgICAgIHBvc2l0aW9uX2VuY29kaW5nID0gUE9TSVRJT05fRU5DT0RJTkcgaWYgcG9zaXRpb25fZW5jb2RpbmcgaXMgTm9uZSBlbHNlIHBvc2l0aW9uX2VuY29kaW5nCiAgICAgICAgcm9wZV9iYXNlID0gUk9QRV9CQVNFIGlmIHJvcGVfYmFzZSBpcyBOb25lIGVsc2Ugcm9wZV9iYXNlCiAgICAgICAgcm9wZV9tYXhfc2VxX2xlbiA9IFJPUEVfTUFYX1NFUV9MRU4gaWYgcm9wZV9tYXhfc2VxX2xlbiBpcyBOb25lIGVsc2Ugcm9wZV9tYXhfc2VxX2xlbgoKICAgICAgICBpZiBlbWJlZF9kaW0gJSBudW1faGVhZHMgIT0gMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZW1iZWRfZGltIG11c3QgYmUgZGl2aXNpYmxlIGJ5IG51bV9oZWFkcyIpCiAgICAgICAgaWYgbnVtX2hlYWRzICUgbnVtX2t2X2hlYWRzICE9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm51bV9oZWFkcyBtdXN0IGJlIGRpdmlzaWJsZSBieSBudW1fa3ZfaGVhZHMiKQogICAgICAgIGlmIG51bV9rdl9oZWFkcyA8PSAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJudW1fa3ZfaGVhZHMgbXVzdCBiZSBwb3NpdGl2ZSIpCgogICAgICAgIHNlbGYuZW1iZWRfZGltID0gZW1iZWRfZGltCiAgICAgICAgc2VsZi5udW1faGVhZHMgPSBudW1faGVhZHMKICAgICAgICBzZWxmLm51bV9rdl9oZWFkcyA9IG51bV9rdl9oZWFkcwogICAgICAgIHNlbGYuaGVhZF9kaW0gPSBlbWJlZF9kaW0gLy8gbnVtX2hlYWRzCiAgICAgICAgc2VsZi5rdl9kaW0gPSBudW1fa3ZfaGVhZHMgKiBzZWxmLmhlYWRfZGltCiAgICAgICAgc2VsZi5kcm9wb3V0X3JhdGUgPSBkcm9wb3V0CiAgICAgICAgc2VsZi5wb3NpdGlvbl9lbmNvZGluZyA9IHBvc2l0aW9uX2VuY29kaW5nCiAgICAgICAgc2VsZi5mdXNlZF9xa3YgPSBmdXNlZF9xa3YKICAgICAgICBzZWxmLnVzZV9zZHBhID0gdXNlX3NkcGEgYW5kIGhhc2F0dHIoRiwgInNjYWxlZF9kb3RfcHJvZHVjdF9hdHRlbnRpb24iKQogICAgICAgIHNlbGYucWtfbm9ybSA9IHFrX25vcm0KICAgICAgICBpZiBzbGlkaW5nX3dpbmRvdyBpcyBub3QgTm9uZSBhbmQgc2xpZGluZ193aW5kb3cgPD0gMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2xpZGluZ193aW5kb3cgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICAgICAgc2VsZi5zbGlkaW5nX3dpbmRvdyA9IHNsaWRpbmdfd2luZG93CgogICAgICAgIGlmIGZ1c2VkX3FrdjoKICAgICAgICAgICAgc2VsZi5xa3ZfcHJvaiA9IExpbmVhcigKICAgICAgICAgICAgICAgIGVtYmVkX2RpbSwgZW1iZWRfZGltICsgMiAqIHNlbGYua3ZfZGltLCBiaWFzPXVzZV9iaWFzCiAgICAgICAgICAgICkKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIFRoaXMgbGF5b3V0IHByZXNlcnZlcyBjb21wYXRpYmlsaXR5IHdpdGggZXhpc3RpbmcgY2hlY2twb2ludHMuCiAgICAgICAgICAgIHNlbGYucV9wcm9qID0gTGluZWFyKGVtYmVkX2RpbSwgZW1iZWRfZGltLCBiaWFzPXVzZV9iaWFzKQogICAgICAgICAgICBzZWxmLmtfcHJvaiA9IExpbmVhcihlbWJlZF9kaW0sIHNlbGYua3ZfZGltLCBiaWFzPXVzZV9iaWFzKQogICAgICAgICAgICBzZWxmLnZfcHJvaiA9IExpbmVhcihlbWJlZF9kaW0sIHNlbGYua3ZfZGltLCBiaWFzPXVzZV9iaWFzKQoKICAgICAgICBzZWxmLm91dF9wcm9qID0gTGluZWFyKGVtYmVkX2RpbSwgZW1iZWRfZGltLCBiaWFzPXVzZV9iaWFzKQogICAgICAgIHNlbGYucmVzaWRfZHJvcG91dCA9IG5uLkRyb3BvdXQoZHJvcG91dCkKCiAgICAgICAgIyBLZXB0IG9ubHkgZm9yIHRoZSBwb3J0YWJsZS9tYW51YWwgYXR0ZW50aW9uIGZhbGxiYWNrLgogICAgICAgIG1hc2sgPSB0b3JjaC50cmlsKHRvcmNoLm9uZXMoY29udGV4dF9sZW5ndGgsIGNvbnRleHRfbGVuZ3RoLCBkdHlwZT10b3JjaC5ib29sKSkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiY2F1c2FsX21hc2siLCBtYXNrLCBwZXJzaXN0ZW50PUZhbHNlKQoKICAgICAgICBzZWxmLnJvcGUgPSAoCiAgICAgICAgICAgIFJvdGFyeUVtYmVkZGluZygKICAgICAgICAgICAgICAgIHNlbGYuaGVhZF9kaW0sIHJvcGVfbWF4X3NlcV9sZW4sIHJvcGVfYmFzZSwgcm9wZV9zY2FsaW5nX2ZhY3RvcgogICAgICAgICAgICApCiAgICAgICAgICAgIGlmIHBvc2l0aW9uX2VuY29kaW5nID09ICJyb3BlIgogICAgICAgICAgICBlbHNlIE5vbmUKICAgICAgICApCgogICAgZGVmIF9wcm9qZWN0KHNlbGYsIHg6IHRvcmNoLlRlbnNvcik6CiAgICAgICAgaWYgc2VsZi5mdXNlZF9xa3Y6CiAgICAgICAgICAgIHByb2plY3RlZCA9IHNlbGYucWt2X3Byb2ooeCkKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLnNwbGl0KAogICAgICAgICAgICAgICAgcHJvamVjdGVkLCBbc2VsZi5lbWJlZF9kaW0sIHNlbGYua3ZfZGltLCBzZWxmLmt2X2RpbV0sIGRpbT0tMQogICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHNlbGYucV9wcm9qKHgpLCBzZWxmLmtfcHJvaih4KSwgc2VsZi52X3Byb2ooeCkKCiAgICBkZWYgX2V4cGFuZF9rdihzZWxmLCB0ZW5zb3I6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIkV4cGFuZCBncm91cGVkIEsvViBoZWFkcyB3aXRob3V0IGFsbG9jYXRpbmcgd2hlbiBwb3NzaWJsZS4iIiIKICAgICAgICBpZiBzZWxmLm51bV9rdl9oZWFkcyA9PSBzZWxmLm51bV9oZWFkczoKICAgICAgICAgICAgcmV0dXJuIHRlbnNvcgogICAgICAgIHJlcGVhdHMgPSBzZWxmLm51bV9oZWFkcyAvLyBzZWxmLm51bV9rdl9oZWFkcwogICAgICAgIGJhdGNoLCBrdl9oZWFkcywgc2VxX2xlbiwgaGVhZF9kaW0gPSB0ZW5zb3Iuc2hhcGUKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICB0ZW5zb3JbOiwgOiwgTm9uZSwgOiwgOl0KICAgICAgICAgICAgLmV4cGFuZChiYXRjaCwga3ZfaGVhZHMsIHJlcGVhdHMsIHNlcV9sZW4sIGhlYWRfZGltKQogICAgICAgICAgICAucmVzaGFwZShiYXRjaCwgc2VsZi5udW1faGVhZHMsIHNlcV9sZW4sIGhlYWRfZGltKQogICAgICAgICkKCiAgICBkZWYgX2F0dGVudGlvbl9tYXNrKAogICAgICAgIHNlbGYsCiAgICAgICAgcXVlcnlfbGVuZ3RoOiBpbnQsCiAgICAgICAga2V5X2xlbmd0aDogaW50LAogICAgICAgIHBvc2l0aW9uX29mZnNldDogaW50LAogICAgICAgIGtleV9zdGFydF9wb3NpdGlvbjogaW50LAogICAgICAgIGRldmljZTogdG9yY2guZGV2aWNlLAogICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgcXVlcnlfcG9zaXRpb25zID0gdG9yY2guYXJhbmdlKAogICAgICAgICAgICBwb3NpdGlvbl9vZmZzZXQsIHBvc2l0aW9uX29mZnNldCArIHF1ZXJ5X2xlbmd0aCwgZGV2aWNlPWRldmljZQogICAgICAgICkKICAgICAgICBrZXlfcG9zaXRpb25zID0gdG9yY2guYXJhbmdlKAogICAgICAgICAgICBrZXlfc3RhcnRfcG9zaXRpb24sIGtleV9zdGFydF9wb3NpdGlvbiArIGtleV9sZW5ndGgsIGRldmljZT1kZXZpY2UKICAgICAgICApCiAgICAgICAgbWFzayA9IGtleV9wb3NpdGlvbnMudW5zcXVlZXplKDApIDw9IHF1ZXJ5X3Bvc2l0aW9ucy51bnNxdWVlemUoMSkKICAgICAgICBpZiBzZWxmLnNsaWRpbmdfd2luZG93IGlzIG5vdCBOb25lOgogICAgICAgICAgICBtYXNrICY9IGtleV9wb3NpdGlvbnMudW5zcXVlZXplKDApID4gKAogICAgICAgICAgICAgICAgcXVlcnlfcG9zaXRpb25zLnVuc3F1ZWV6ZSgxKSAtIHNlbGYuc2xpZGluZ193aW5kb3cKICAgICAgICAgICAgKQogICAgICAgIHJldHVybiBtYXNrCgogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICB4OiB0b3JjaC5UZW5zb3IsCiAgICAgICAga3ZfY2FjaGU6IE9wdGlvbmFsW0tWQ2FjaGVdID0gTm9uZSwKICAgICAgICB1c2VfY2FjaGU6IGJvb2wgPSBGYWxzZSwKICAgICk6CiAgICAgICAgYmF0Y2hfc2l6ZSwgcXVlcnlfbGVuZ3RoLCBfID0geC5zaGFwZQogICAgICAgIGNhY2hlZF9sZW5ndGggPSAwIGlmIGt2X2NhY2hlIGlzIE5vbmUgZWxzZSBrdl9jYWNoZVswXS5zaXplKDIpCiAgICAgICAgcG9zaXRpb25fb2Zmc2V0ID0gKAogICAgICAgICAgICAwCiAgICAgICAgICAgIGlmIGt2X2NhY2hlIGlzIE5vbmUKICAgICAgICAgICAgZWxzZSBpbnQoa3ZfY2FjaGVbMl0pIGlmIGxlbihrdl9jYWNoZSkgPiAyIGVsc2UgY2FjaGVkX2xlbmd0aAogICAgICAgICkKCiAgICAgICAgcSwgaywgdiA9IHNlbGYuX3Byb2plY3QoeCkKICAgICAgICBxID0gcS52aWV3KGJhdGNoX3NpemUsIHF1ZXJ5X2xlbmd0aCwgc2VsZi5udW1faGVhZHMsIHNlbGYuaGVhZF9kaW0pLnRyYW5zcG9zZSgxLCAyKQogICAgICAgIGsgPSBrLnZpZXcoYmF0Y2hfc2l6ZSwgcXVlcnlfbGVuZ3RoLCBzZWxmLm51bV9rdl9oZWFkcywgc2VsZi5oZWFkX2RpbSkudHJhbnNwb3NlKDEsIDIpCiAgICAgICAgdiA9IHYudmlldyhiYXRjaF9zaXplLCBxdWVyeV9sZW5ndGgsIHNlbGYubnVtX2t2X2hlYWRzLCBzZWxmLmhlYWRfZGltKS50cmFuc3Bvc2UoMSwgMikKCiAgICAgICAgaWYgc2VsZi5yb3BlIGlzIG5vdCBOb25lOgogICAgICAgICAgICBxLCBrID0gc2VsZi5yb3BlKAogICAgICAgICAgICAgICAgcSwgaywgc2VxX2xlbj1xdWVyeV9sZW5ndGgsIHBvc2l0aW9uX29mZnNldD1wb3NpdGlvbl9vZmZzZXQKICAgICAgICAgICAgKQoKICAgICAgICBpZiBzZWxmLnFrX25vcm06CiAgICAgICAgICAgICMgVW5pdCBSTVMga2VlcHMgZG90IHByb2R1Y3RzIGNvbnRyb2xsZWQgd2hpbGUgcHJlc2VydmluZyBTRFBBIHNjYWxpbmcuCiAgICAgICAgICAgIHEgPSBGLm5vcm1hbGl6ZShxLmZsb2F0KCksIGRpbT0tMSkudG8ocS5kdHlwZSkgKiBzZWxmLmhlYWRfZGltKiowLjUKICAgICAgICAgICAgayA9IEYubm9ybWFsaXplKGsuZmxvYXQoKSwgZGltPS0xKS50byhrLmR0eXBlKSAqIHNlbGYuaGVhZF9kaW0qKjAuNQoKICAgICAgICBpZiBrdl9jYWNoZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgY2FjaGVkX2ssIGNhY2hlZF92ID0ga3ZfY2FjaGVbOjJdCiAgICAgICAgICAgIGsgPSB0b3JjaC5jYXQoKGNhY2hlZF9rLCBrKSwgZGltPTIpCiAgICAgICAgICAgIHYgPSB0b3JjaC5jYXQoKGNhY2hlZF92LCB2KSwgZGltPTIpCiAgICAgICAgaWYgKAogICAgICAgICAgICBzZWxmLnNsaWRpbmdfd2luZG93IGlzIG5vdCBOb25lCiAgICAgICAgICAgIGFuZCBxdWVyeV9sZW5ndGggPT0gMQogICAgICAgICAgICBhbmQgay5zaXplKDIpID4gc2VsZi5zbGlkaW5nX3dpbmRvdwogICAgICAgICk6CiAgICAgICAgICAgIGsgPSBrWzosIDosIC1zZWxmLnNsaWRpbmdfd2luZG93IDpdCiAgICAgICAgICAgIHYgPSB2WzosIDosIC1zZWxmLnNsaWRpbmdfd2luZG93IDpdCiAgICAgICAgcHJlc2VudCA9IE5vbmUKICAgICAgICBpZiB1c2VfY2FjaGU6CiAgICAgICAgICAgIHByZXNlbnRfaywgcHJlc2VudF92ID0gaywgdgogICAgICAgICAgICBpZiBzZWxmLnNsaWRpbmdfd2luZG93IGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcHJlc2VudF9rID0gcHJlc2VudF9rWzosIDosIC1zZWxmLnNsaWRpbmdfd2luZG93IDpdCiAgICAgICAgICAgICAgICBwcmVzZW50X3YgPSBwcmVzZW50X3ZbOiwgOiwgLXNlbGYuc2xpZGluZ193aW5kb3cgOl0KICAgICAgICAgICAgcHJlc2VudCA9IChwcmVzZW50X2ssIHByZXNlbnRfdiwgcG9zaXRpb25fb2Zmc2V0ICsgcXVlcnlfbGVuZ3RoKQoKICAgICAgICBleHBhbmRlZF9rID0gc2VsZi5fZXhwYW5kX2t2KGspCiAgICAgICAgZXhwYW5kZWRfdiA9IHNlbGYuX2V4cGFuZF9rdih2KQogICAgICAgICMgUHlUb3JjaC9YTEEgYXV0b2Nhc3QgY2FuIGxlYXZlIFEvSyBpbiBmbG9hdDMyIHdoaWxlIFYgaXMgYmZsb2F0MTYuCiAgICAgICAgIyBTRFBBIHJlcXVpcmVzIGFsbCB0aHJlZSBvcGVyYW5kcyB0byBzaGFyZSBhIGR0eXBlLgogICAgICAgIGlmIGV4cGFuZGVkX2suZHR5cGUgIT0gcS5kdHlwZToKICAgICAgICAgICAgZXhwYW5kZWRfayA9IGV4cGFuZGVkX2sudG8ocS5kdHlwZSkKICAgICAgICBpZiBleHBhbmRlZF92LmR0eXBlICE9IHEuZHR5cGU6CiAgICAgICAgICAgIGV4cGFuZGVkX3YgPSBleHBhbmRlZF92LnRvKHEuZHR5cGUpCiAgICAgICAga2V5X2xlbmd0aCA9IGV4cGFuZGVkX2suc2l6ZSgyKQogICAgICAgIGRyb3BvdXRfcCA9IHNlbGYuZHJvcG91dF9yYXRlIGlmIHNlbGYudHJhaW5pbmcgZWxzZSAwLjAKCiAgICAgICAgaWYgc2VsZi51c2Vfc2RwYToKICAgICAgICAgICAgaWYgcG9zaXRpb25fb2Zmc2V0ID09IDAgYW5kIHNlbGYuc2xpZGluZ193aW5kb3cgaXMgTm9uZToKICAgICAgICAgICAgICAgIG91dHB1dCA9IEYuc2NhbGVkX2RvdF9wcm9kdWN0X2F0dGVudGlvbigKICAgICAgICAgICAgICAgICAgICBxLCBleHBhbmRlZF9rLCBleHBhbmRlZF92LCBkcm9wb3V0X3A9ZHJvcG91dF9wLCBpc19jYXVzYWw9VHJ1ZQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbGlmIHF1ZXJ5X2xlbmd0aCA9PSAxOgogICAgICAgICAgICAgICAgIyBEdXJpbmcgdG9rZW4tYnktdG9rZW4gZGVjb2RpbmcgZXZlcnkgcmV0YWluZWQgY2FjaGUga2V5IGlzIGluCiAgICAgICAgICAgICAgICAjIHRoZSBwYXN0IChvciBpcyB0aGUgY3VycmVudCB0b2tlbiksIHNvIG5vIG1hc2sgaXMgbmVjZXNzYXJ5LgogICAgICAgICAgICAgICAgb3V0cHV0ID0gRi5zY2FsZWRfZG90X3Byb2R1Y3RfYXR0ZW50aW9uKAogICAgICAgICAgICAgICAgICAgIHEsCiAgICAgICAgICAgICAgICAgICAgZXhwYW5kZWRfaywKICAgICAgICAgICAgICAgICAgICBleHBhbmRlZF92LAogICAgICAgICAgICAgICAgICAgIGRyb3BvdXRfcD1kcm9wb3V0X3AsCiAgICAgICAgICAgICAgICAgICAgaXNfY2F1c2FsPUZhbHNlLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAga2V5X3N0YXJ0ID0gcG9zaXRpb25fb2Zmc2V0IC0gY2FjaGVkX2xlbmd0aAogICAgICAgICAgICAgICAgbWFzayA9IHNlbGYuX2F0dGVudGlvbl9tYXNrKAogICAgICAgICAgICAgICAgICAgIHF1ZXJ5X2xlbmd0aCwga2V5X2xlbmd0aCwgcG9zaXRpb25fb2Zmc2V0LCBrZXlfc3RhcnQsIHguZGV2aWNlCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBvdXRwdXQgPSBGLnNjYWxlZF9kb3RfcHJvZHVjdF9hdHRlbnRpb24oCiAgICAgICAgICAgICAgICAgICAgcSwKICAgICAgICAgICAgICAgICAgICBleHBhbmRlZF9rLAogICAgICAgICAgICAgICAgICAgIGV4cGFuZGVkX3YsCiAgICAgICAgICAgICAgICAgICAgYXR0bl9tYXNrPW1hc2ssCiAgICAgICAgICAgICAgICAgICAgZHJvcG91dF9wPWRyb3BvdXRfcCwKICAgICAgICAgICAgICAgICAgICBpc19jYXVzYWw9RmFsc2UsCiAgICAgICAgICAgICAgICApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2NvcmVzID0gcSBAIGV4cGFuZGVkX2sudHJhbnNwb3NlKC0yLCAtMSkgLyBzZWxmLmhlYWRfZGltKiowLjUKICAgICAgICAgICAga2V5X3N0YXJ0ID0gcG9zaXRpb25fb2Zmc2V0IC0gY2FjaGVkX2xlbmd0aAogICAgICAgICAgICBtYXNrID0gc2VsZi5fYXR0ZW50aW9uX21hc2soCiAgICAgICAgICAgICAgICBxdWVyeV9sZW5ndGgsIGtleV9sZW5ndGgsIHBvc2l0aW9uX29mZnNldCwga2V5X3N0YXJ0LCB4LmRldmljZQogICAgICAgICAgICApCiAgICAgICAgICAgIHNjb3JlcyA9IHNjb3Jlcy5tYXNrZWRfZmlsbCh+bWFzaywgZmxvYXQoIi1pbmYiKSkKICAgICAgICAgICAgd2VpZ2h0cyA9IEYuc29mdG1heChzY29yZXMuZmxvYXQoKSwgZGltPS0xKS50byhxLmR0eXBlKQogICAgICAgICAgICB3ZWlnaHRzID0gRi5kcm9wb3V0KHdlaWdodHMsIGRyb3BvdXRfcCwgdHJhaW5pbmc9c2VsZi50cmFpbmluZykKICAgICAgICAgICAgb3V0cHV0ID0gd2VpZ2h0cyBAIGV4cGFuZGVkX3YKCiAgICAgICAgb3V0cHV0ID0gb3V0cHV0LnRyYW5zcG9zZSgxLCAyKS5jb250aWd1b3VzKCkudmlldygKICAgICAgICAgICAgYmF0Y2hfc2l6ZSwgcXVlcnlfbGVuZ3RoLCBzZWxmLmVtYmVkX2RpbQogICAgICAgICkKICAgICAgICBvdXRwdXQgPSBzZWxmLnJlc2lkX2Ryb3BvdXQoc2VsZi5vdXRfcHJvaihvdXRwdXQpKQogICAgICAgIHJldHVybiAob3V0cHV0LCBwcmVzZW50KSBpZiB1c2VfY2FjaGUgZWxzZSBvdXRwdXQK"}')
for relative, encoded in embedded.items():
    destination = PROJECT_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))

os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tokenizers>=0.13', 'datasets>=2.14', 'kagglehub>=0.3',
                'tensorboard>=2.14', 'tqdm>=4.65', 'pyyaml>=6'], check=True)
print('[OK] Project:', PROJECT_ROOT)


In [ ]:
import importlib.util
if importlib.util.find_spec('torch_xla') is None:
    raise RuntimeError(
        'torch_xla is unavailable. This Kaggle run was not launched with a TPU accelerator.'
    )

# Do not import torch_xla or query TPU devices here. PyTorch/XLA requires the
# training launcher to be the first process that initializes the TPU runtime.
print('torch_xla package found; TPU initialization deferred to train_xla.py')


In [ ]:
# Resolve all 72 binary splits and sidecars from attached Kaggle Datasets.
expected_registry = json.loads((PROJECT_ROOT / 'configs/datasets_v3_8b.json').read_text())
input_root = Path('/kaggle/input')

def attached(name):
    matches = [p for p in input_root.rglob(name) if p.is_file() and p.stat().st_size > 0]
    return max(matches, key=lambda p: p.stat().st_size) if matches else None

runtime_registry = {}
missing = []
for dataset_name, entry in expected_registry.items():
    runtime_entry = {'weight': entry['weight']}
    for split in ('train', 'val'):
        filename = Path(str(entry[split]).replace(chr(92), '/')).name
        binary = attached(filename)
        sidecar = attached(filename + '.meta.json')
        if binary is None or sidecar is None or sidecar.parent != binary.parent:
            missing.append(filename)
        else:
            runtime_entry[split] = str(binary)
    runtime_registry[dataset_name] = runtime_entry
if missing:
    raise FileNotFoundError(f'Missing binaries/sidecars ({len(missing)}): {sorted(set(missing))}')
runtime_registry_path = CONFIG_ROOT / 'datasets_v3_8b_kaggle.json'
runtime_registry_path.write_text(json.dumps(runtime_registry, indent=2) + '\n')
print(f'[OK] Resolved {len(runtime_registry)} dataset sources.')


In [ ]:
# Preserve the CUDA run's effective global batch: 1 * 8 * 4 = 32.
PER_CORE_BATCH = 1
GRAD_ACCUM_STEPS = 4
EXPECTED_TPU_CORES = 8
if PER_CORE_BATCH * EXPECTED_TPU_CORES * GRAD_ACCUM_STEPS != 32:
    raise ValueError('The effective global batch must remain 32.')
config = json.loads((PROJECT_ROOT / 'configs/train_config_v3_2xt4.json').read_text())
config['model']['gradient_checkpointing'] = False
config['training'].update({
    'batch_size': PER_CORE_BATCH, 'grad_accum_steps': GRAD_ACCUM_STEPS,
    'planned_world_size': EXPECTED_TPU_CORES,
    'amp_dtype': 'bfloat16', 'torch_compile': False, 'log_interval': 10,
})
config['data'].update({
    'datasets_file': str(runtime_registry_path), 'batch_size': PER_CORE_BATCH,
    'num_workers': 0, 'shuffle': False,
})
config['checkpoint'].update({
    'checkpoint_dir': str(CHECKPOINT_ROOT),
    'milestone_dir': str(CHECKPOINT_ROOT / 'milestones'),
    'metrics_file': str(LOG_ROOT / 'metrics.jsonl'),
    'log_dir': str(LOG_ROOT), 'tensorboard_dir': str(LOG_ROOT / 'tensorboard'),
    'save_interval': 1000, 'log_interval': 10,
    'backup': {
        'enabled': True, 'provider': 'kaggle_dataset',
        'handle': 'aethyx/aethyxlm-v3-live-checkpoints',
        'required': True, 'retries': 3,
    },
})
runtime_config = CONFIG_ROOT / 'train_config_v3_tpu_runtime.json'
runtime_config.write_text(json.dumps(config, indent=2) + '\n')
print('[OK] Runtime config:', runtime_config)


In [ ]:
# Select the highest numbered compatible checkpoint, or start v3 from scratch.
RESUME_CHECKPOINT = None
roots = [CHECKPOINT_ROOT, Path('/kaggle/input')]
candidates = []
for root in roots:
    if root.exists():
        candidates.extend(root.rglob('checkpoint_step_*.pt'))
numbered = []
for path in candidates:
    match = re.fullmatch(r'checkpoint_step_(\d+)\.pt', path.name)
    if match and path.stat().st_size > 10 * 2**20:
        numbered.append((int(match.group(1)), path))
resume_path = Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else (
    max(numbered, key=lambda item: item[0])[1] if numbered else None
)
resume_args = [] if resume_path is None else ['--resume', str(resume_path)]
print('[OK] Starting fresh v3 run.' if resume_path is None else f'[OK] Resuming: {resume_path}')

subprocess.run([sys.executable, 'scripts/check_training_readiness.py',
                '--config', str(runtime_config)], cwd=PROJECT_ROOT, check=True)


In [ ]:
command = [
    sys.executable, 'train_xla.py', '--config', str(runtime_config),
] + resume_args
print('Running:', ' '.join(command))
started = time.time()
process = subprocess.Popen(
    command, cwd=PROJECT_ROOT, env=os.environ.copy(), start_new_session=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    os.killpg(process.pid, signal.SIGINT)
    return_code = process.wait(timeout=300)
print(f'Exit={return_code}; elapsed={(time.time() - started) / 3600:.2f}h')
if return_code not in (0, 130, -signal.SIGINT):
    raise RuntimeError(f'Training failed with exit code {return_code}.')


In [ ]:
checkpoints = sorted(CHECKPOINT_ROOT.glob('*.pt'), key=lambda p: p.stat().st_mtime)
for path in checkpoints:
    print(f'{path.name:36s} {path.stat().st_size / 2**20:8.1f} MiB')
print('Metrics:', LOG_ROOT / 'metrics.jsonl')
print('Persistent backup: aethyx/aethyxlm-v3-live-checkpoints')
